# **GPCR ファミリーの系統樹を作ろう**
## アミノ酸配列から受容体スーパーファミリーの進化を読み解く

このノートブックでは、ヒトの **G タンパク質共役受容体 (GPCR) 25 種** のアミノ酸配列から、
タンパク質どうしの進化的な近さを表す **系統樹 (phylogenetic tree)** を作ります。

DNA を扱った鳥類・哺乳類のノートブックとの一番の違いは、**タンパク質(アミノ酸20文字)を直接比べる**点と、
**配列の長さが大きくバラついている**点です。この違いが解析手法の選び方にどう効いてくるかに注目してください。

解析を 2 段構えで行うのがポイントです。

1. **入門編** — Python だけで完結するアプローチ(ペアワイズアラインメント + 近隣結合法)で大まかな関係をつかむ
2. **本格編** — 研究現場の業界標準 **MAFFT** で多重整列し、**IQ-TREE** の **最尤法 (Maximum Likelihood)** + ブートストラップで信頼度付きの系統樹を作る
3. **発展編** — 配列ではなく **立体構造**(AlphaFold モデル)を直接重ね合わせて、**構造に基づく系統樹**を作り、配列の樹と比べる

### 学習目標

1. タンパク質配列データ(FASTA形式)を扱えるようになる
2. **配列の類似度** と **遺伝的距離** の関係を理解する
3. **多重整列 (MSA)** がなぜ必要か、長さの違う配列でどう効くかを体感する
4. **近隣結合法 (NJ法)** と **最尤法 (ML法)** の違い、**ブートストラップ** による信頼度を知る
5. 作った樹が **GPCR のクラス分類 (A/B/C/F)** や **機能グループ** を再現できるか考察する

### GPCR とは

**G タンパク質共役受容体 (G Protein-Coupled Receptor, GPCR)** は、細胞膜を **7 回貫通** する受容体ファミリーです。

- 光・におい・ホルモン・神経伝達物質など、外界の多様なシグナルを細胞内に伝える「アンテナ」
- ヒトゲノムに **約 800 種** が存在し、現在の医薬品の **およそ 1/3** がこのファミリーを標的にする
- すべて「7回膜貫通」という共通の骨格を持つが、配列としては大きく分岐している

> このノートブックは大学1年生の一般教養課程を想定し、専門用語は最小限にしています。
> ライセンス:配列・構造データは UniProt / NCBI / AlphaFold DB のパブリックデータ(`gpcr/gpcr_list.tsv` 参照)。

### GPCR のクラス分類

GPCR は配列の似かたから、いくつかの **クラス** に分けられます(GRAFS 分類などが有名)。
このデータセットには代表的な 4 クラスが含まれています。

| クラス | 別名 | 例 | 特徴 |
| --- | --- | --- | --- |
| **A** | Rhodopsin 様 | ロドプシン、アドレナリン受容体、ドーパミン受容体 | 最大のグループ(ヒト GPCR の約 85%)。膜貫通領域以外は短め |
| **B** | Secretin 様 | グルカゴン受容体、GLP-1 受容体 | ペプチドホルモンを受容。大きな細胞外ドメインを持つ |
| **C** | Glutamate 様 | 代謝型グルタミン酸受容体、カルシウム感知受容体 | **非常に大きな細胞外ドメイン**でリガンドを捕まえる |
| **F** | Frizzled | Smoothened、Frizzled-1 | Wnt/Hedgehog シグナル。発生に重要 |

**問い:** 配列だけから系統樹を作ったとき、この A/B/C/F の区分は再現されるでしょうか?
そして、同じクラス A の中でも「光を受容するオプシン」「アミンを受容する受容体」などの **機能グループ** は分かれて見えるでしょうか?

## 0. 必要ライブラリのインストール(初回のみ)

Python パッケージをインストールします。

- `biopython` : 配列解析と系統樹計算
- `matplotlib-fontja` : 日本語フォント

**MAFFT と IQ-TREE は別途インストールが必要です(後のセクション 10 で扱います)。**

In [ ]:
%pip install biopython pandas numpy matplotlib seaborn matplotlib-fontja

## 1. ライブラリの読み込みと日本語フォントの設定

グラフに日本語を出すために `matplotlib-fontja` を使います。
**順序が重要**:seaborn のスタイル設定を先に行い、その後に `matplotlib_fontja` を読み込みます。

In [ ]:
import sys
import shutil
import subprocess
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns

from Bio import SeqIO, AlignIO, Phylo
from Bio.Align import PairwiseAligner, substitution_matrices
from Bio.Phylo.TreeConstruction import DistanceMatrix, DistanceTreeConstructor

# 日本語フォントの設定(順序が重要)
sns.set_style("whitegrid")
print("kernel python :", sys.executable)
try:
    import matplotlib_fontja  # noqa: F401
    _sans = [f for f in plt.rcParams["font.sans-serif"] if f != "IPAexGothic"]
    plt.rcParams["font.sans-serif"] = ["IPAexGothic"] + _sans
    print("matplotlib-fontja: OK")
except ImportError:
    print("matplotlib-fontja が見つかりません。上のセルでインストールしてください。")
plt.rcParams["axes.unicode_minus"] = False

# データファイルへのパス(実行ディレクトリの違いに強いよう自動検出)
def _find_dir(candidates):
    for c in candidates:
        if Path(c).exists():
            return Path(c)
    raise FileNotFoundError(f"gpcr フォルダが見つかりません。候補: {candidates}")

GPCR_DIR = _find_dir(["gpcr", "../gpcr", "phylogenetics/gpcr"])
FASTA_PATH = GPCR_DIR / "all_GPCRs.fasta"
LIST_PATH = GPCR_DIR / "gpcr_list.tsv"
PDB_DIR = GPCR_DIR / "pdb"

# MAFFT / IQ-TREE 用の作業ディレクトリ
WORK_DIR = Path("./phylo_work_gpcr")
WORK_DIR.mkdir(exist_ok=True)

for p in [FASTA_PATH, LIST_PATH]:
    assert p.exists(), f"見つかりません: {p}"
print(f"データ準備 OK: {GPCR_DIR.resolve()}")
print(f"作業ディレクトリ: {WORK_DIR.resolve()}")

## 2. FASTA ファイルの読み込み

**FASTA** は配列データを表す最も基本的なテキスト形式です。今回はアミノ酸配列(1文字コード)です。

```
>RHO|P08100|ClassA|Rhodopsin|RefSeq:NP_000530.1   ← ヘッダー(`>` で始まる)
MNGTEGPNFYVPFSNATGVVRSPFEYPQYYLAEPWQFSMLAAYMFLLI...  ← 配列本体(20種類のアミノ酸)
```

ヘッダーは `>遺伝子名|UniProt番号|クラス|説明|RefSeq` を `|` で区切った形式です。
ここから **遺伝子名** を ID として取り出し、**クラス** をあとで色分けに使います。

In [ ]:
records = list(SeqIO.parse(FASTA_PATH, "fasta"))
print(f"読み込んだ配列数: {len(records)}")
print(f"最初のヘッダー : {records[0].description}")
print(f"最初の60残基   : {str(records[0].seq)[:60]} ...")

# ヘッダーを分解して「遺伝子名」を ID に、クラスを別途記録
gene_to_class = {}
for rec in records:
    parts = rec.description.split("|")
    gene = parts[0]
    cls = parts[2].replace("Class", "")   # "ClassA" -> "A"
    gene_to_class[gene] = cls
    rec.id = gene
    rec.name = gene
    rec.description = ""

print(f"\n整理後の ID 一覧: {[r.id for r in records]}")

## 3. メタデータ(遺伝子名・クラス・説明)の読み込み

各受容体の遺伝子名・UniProt 番号・クラス・説明が `gpcr_list.tsv` にまとめられています。

In [ ]:
metadata = pd.read_csv(LIST_PATH, sep="\t")
# 先頭が "# Gene" のようにコメント記号付きなので列名を整える
metadata.columns = [c.lstrip("# ").strip() for c in metadata.columns]

print(f"メタデータ行数: {len(metadata)}")
print("\n各クラスの受容体数:")
print(metadata["Class"].value_counts().sort_index())
metadata.head()

## 4. 配列の長さを確認 — なぜタンパク質では「切り詰め」が使えないか

解析に入る前にデータの「健康状態」をチェックします。DNA(cytb)の解析では全配列がほぼ同じ長さ(約1140塩基)
だったので「先頭から共通の長さだけ切り出す」という乱暴な近似が許されました。

GPCR ではどうでしょうか? **クラスごとに配列長を比べてみます。**

In [ ]:
seq_stats = pd.DataFrame({
    "gene": [r.id for r in records],
    "length": [len(r.seq) for r in records],
})
seq_stats["Class"] = seq_stats["gene"].map(gene_to_class)
seq_stats = seq_stats.merge(metadata[["Gene", "Description"]],
                            left_on="gene", right_on="Gene", how="left").drop(columns="Gene")

print("配列長の統計:")
print(seq_stats["length"].describe().round(1))
print(f"\n最短: {seq_stats.loc[seq_stats['length'].idxmin(), 'gene']} "
      f"({seq_stats['length'].min()} aa)")
print(f"最長: {seq_stats.loc[seq_stats['length'].idxmax(), 'gene']} "
      f"({seq_stats['length'].max()} aa)")
seq_stats.sort_values("length").head()

In [ ]:
# クラス別の配列長(箱ひげ図 + 個々の点)
class_order = ["A", "B", "C", "F"]
fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(data=seq_stats, x="Class", y="length", order=class_order,
            color="lightsteelblue", ax=ax)
sns.stripplot(data=seq_stats, x="Class", y="length", order=class_order,
              color="darkblue", alpha=0.7, size=5, ax=ax)
ax.set_xlabel("GPCR クラス")
ax.set_ylabel("配列長 (アミノ酸残基数)")
ax.set_title("GPCR のアミノ酸配列長(クラス別)")
plt.tight_layout()
plt.show()

**観察ポイント**

- 配列長は **348 残基(オプシン)から 1078 残基(カルシウム感知受容体)まで** 3 倍以上の開きがある
- クラス **C**(グルタミン酸様)は **巨大な細胞外ドメイン**を持つため特に長い
- 共通しているのは中央の「7回膜貫通領域」だけで、その前後の長さはバラバラ

> つまり「先頭から N 残基を切り出す」と、ある受容体の膜貫通領域と別の受容体の細胞外ドメインを
> 比べてしまい、**まったく無意味な比較**になります。
> タンパク質では「**同じ進化的起源の残基を縦に揃える**」=**アラインメント**が必須なのです。

## 5. 入門編 — ペアワイズアラインメントで距離行列を作る

多重整列(全配列をいっぺんに揃える)は少し高度なので、入門編ではより素朴な方法を使います。

> **アイデア:** 2 配列ずつ「ペアワイズアラインメント」して似ている割合(% 同一性)を測り、
> その「似ていなさ」を距離とする。距離法(NJ 法)は **距離行列さえあれば** 系統樹を作れるので、
> 全配列を一度に揃えなくてもよいのです。

ここでは Biopython の `PairwiseAligner` を使い、タンパク質用のスコア表 **BLOSUM62**
(似たアミノ酸どうしの置換は軽いペナルティ)で大域アラインメントします。

$$
d(A, B) = 1 - \frac{\text{一致した残基数}}{\text{ギャップを除いた整列長}}
$$

> 25 配列 → 300 ペアの計算で、数秒〜十数秒かかります。

In [ ]:
aligner = PairwiseAligner()
aligner.mode = "global"
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.open_gap_score = -11      # ギャップを開けるペナルティ(BLOSUM62 の標準設定)
aligner.extend_gap_score = -1     # ギャップを伸ばすペナルティ

def identity_distance(s1, s2):
    # 2配列を大域アラインメントし、1 - (%同一性) を距離として返す
    aln = aligner.align(s1, s2)[0]
    a, b = aln[0], aln[1]
    ident = aln_len = 0
    for x, y in zip(a, b):
        if x == "-" or y == "-":
            continue           # ギャップ列は分母から除く
        aln_len += 1
        if x == y:
            ident += 1
    return 1.0 - ident / aln_len if aln_len else 1.0

names = [r.id for r in records]
seqs = {r.id: str(r.seq) for r in records}

print("ペアワイズ距離を計算中(数秒〜十数秒)...")
# Biopython の DistanceMatrix は下三角(対角含む)で持つ
lower = [[0.0] * (i + 1) for i in range(len(names))]
for i in range(len(names)):
    for j in range(i):
        lower[i][j] = identity_distance(seqs[names[i]], seqs[names[j]])
dm = DistanceMatrix(names, lower)
print("完了。距離行列のサイズ:", len(dm.names), "×", len(dm.names))

print("\n例:")
print(f"  RHO vs OPN1SW (どちらもオプシン, クラスA) d = {dm['RHO', 'OPN1SW']:.3f}")
print(f"  ADRB1 vs ADRB2 (β1/β2 アドレナリン受容体) d = {dm['ADRB1', 'ADRB2']:.3f}")
print(f"  RHO vs CASR  (クラスA vs クラスC)        d = {dm['RHO', 'CASR']:.3f}")
print(f"  SMO vs FZD1  (どちらもクラスF)            d = {dm['SMO', 'FZD1']:.3f}")

## 6. 距離行列をヒートマップで可視化

行と列を **クラス順 (A→B→C→F)** に並べると、同じクラスの受容体どうしが
「ブロック」を成して見えるはずです。

In [ ]:
mat = np.array([[dm[a, b] for b in names] for a in names])
dist_df = pd.DataFrame(mat, index=names, columns=names)

# クラス順 → クラス内は遺伝子名順に並べ替え
ordered = sorted(names, key=lambda g: (class_order.index(gene_to_class[g]), g))
dist_df = dist_df.loc[ordered, ordered]

# ラベルに「遺伝子名 (クラス)」を表示
labels = [f"{g} ({gene_to_class[g]})" for g in dist_df.index]

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(dist_df, cmap="viridis_r",
            xticklabels=labels, yticklabels=labels,
            cbar_kws={"label": "距離 (1 - %同一性)"}, ax=ax, square=True)
ax.set_title("25 受容体間のアミノ酸配列距離(クラス順に並べ替え)")
plt.tight_layout()
plt.show()

**観察ポイント**

- 対角線(自分自身)は距離 0
- 左上の大きなブロック = クラス **A**。さらにその中で小さな濃いブロック(機能グループ)が見える
- クラス **C**(GRM1, CASR)は他のすべてから距離が大きく、明るい行・列として目立つ
- 配列としては GPCR どうしでも **半分以上が違う**(距離 0.5 以上)ことが多い — それでも共通の祖先を持つ

## 7. 近隣結合法 (NJ法) で系統樹を作る

**Neighbor Joining (NJ) 法** は距離行列から系統樹を作る代表的な方法です。
1987年に斎藤・根井(日本人研究者)により発表され、今も広く使われています。

**アルゴリズムのアイデア(直感)**

1. 距離行列をもとに「**最も近い 2 つ**」を選んで枝でつなぐ
2. その 2 つを 1 グループにまとめ、他からの距離を更新
3. すべてが 1 つの木に統合されるまで繰り返す

**根 (root) の決め方:** 鳥類の解析ではワニという明確な **外群** がありましたが、
GPCR は 4 クラスがどれも大きく分岐していて「どれが一番外側の祖先か」が自明ではありません。
そこで今回は **中点ルーティング (midpoint rooting)** ── 最も離れた 2 葉のちょうど中間を根にする ── を使います。

In [ ]:
constructor = DistanceTreeConstructor()
nj_tree = constructor.nj(dm)

# 明確な外群がないので中点で根づけ
nj_tree.root_at_midpoint()

# 内部ノードのデフォルト名(Inner1, ...)はラベル表示の邪魔なので消す
for clade in nj_tree.get_nonterminals():
    clade.name = None

print("NJ 系統樹を作成しました")
print(f"  末端ノード(受容体)の数: {nj_tree.count_terminals()}")

## 8. 系統樹の描画(クラスで色分け)

末端ラベルを **クラス別の色** で塗り分けて描画します。
NJ 樹が GPCR の A/B/C/F クラスをどれくらい再現できているか見てみましょう。

In [ ]:
# クラスごとの色
class_to_color = {
    "A": "#1b9e77",   # 緑(Rhodopsin 様)
    "B": "#d95f02",   # オレンジ(Secretin 様)
    "C": "#7570b3",   # 紫(Glutamate 様)
    "F": "#e7298a",   # ピンク(Frizzled)
}
class_name_ja = {"A": "クラスA (Rhodopsin様)", "B": "クラスB (Secretin様)",
                 "C": "クラスC (Glutamate様)", "F": "クラスF (Frizzled)"}
desc_map = metadata.set_index("Gene")["Description"].to_dict()

def label_func(clade):
    if clade.is_terminal():
        return f"{clade.name}  ({desc_map.get(clade.name, '')})"
    return ""

def gene_color(label):
    if not label or not label.strip():
        return "#000000"
    gene = label.split()[0]
    return class_to_color.get(gene_to_class.get(gene, ""), "#000000")

def draw_class_tree(tree, title, xlabel="距離 (1 - %同一性)"):
    fig, ax = plt.subplots(figsize=(11, 11))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        Phylo.draw(tree, label_func=label_func, do_show=False, axes=ax,
                   label_colors=gene_color)
    ax.set_title(title, fontsize=13)
    handles = [Patch(facecolor=c, label=class_name_ja[k])
               for k, c in class_to_color.items()]
    ax.legend(handles=handles, loc="lower right", framealpha=0.9, fontsize=10)
    plt.tight_layout()
    plt.show()

draw_class_tree(nj_tree, "GPCR 25種の NJ 系統樹(中点ルーティング、クラスで色分け)")

**観察ポイント**

- クラス **B**(GCGR, GLP1R, PTH1R)・クラス **F**(SMO, FZD1)・クラス **C**(GRM1, CASR)が
  それぞれまとまっていれば、配列だけからクラス分類が再現できたことになる
- クラス **A** の中でも、**オプシン**(RHO/OPN1SW/OPN1MW)や
  **アミン受容体**(ADRB/ADRA/DRD/HTR/CHRM/HRH)が小さな枝でまとまっているはず
- NJ 法は速くて手軽だが、配列が大きく分岐していると一部の枝を取り違えることがある(後で ML 法と比較)

## 9. 別の距離法(UPGMA)と比較してみる

**UPGMA** は「距離が最も近いペアを順次まとめる」だけのさらにシンプルな方法です。
ただし **すべての系統で進化速度が一定**という強い仮定を置くため、
進化速度がバラつくタンパク質では NJ より精度が落ちがちです。並べて観察してみましょう。

In [ ]:
upgma_tree = constructor.upgma(dm)
for clade in upgma_tree.get_nonterminals():
    clade.name = None

fig, axes = plt.subplots(1, 2, figsize=(20, 11))
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    Phylo.draw(nj_tree, label_func=label_func, do_show=False, axes=axes[0],
               label_colors=gene_color)
    axes[0].set_title("NJ (Neighbor Joining)")
    Phylo.draw(upgma_tree, label_func=label_func, do_show=False, axes=axes[1],
               label_colors=gene_color)
    axes[1].set_title("UPGMA")
plt.tight_layout()
plt.show()

**比較のポイント**

- UPGMA は **すべての末端が同じ深さに揃う**(超距離木)。進化速度が一定という仮定の表れ
- NJ は **枝の長さがバラつく** ── 進化速度の違いを反映できる
- 大枠のクラス分け(A / B / C / F)が両者で一致していれば、それは信頼できるシグナル

## 10. 本格編 — MAFFT で多重整列し、IQ-TREE で最尤系統樹を推定

ここからは研究現場の **業界標準ツール** で同じデータを解析し直します。

| ツール | 役割 | 入門編との違い |
| --- | --- | --- |
| **MAFFT** | 多重整列 (MSA) | 全配列を **一度に**揃え、必要に応じてギャップを入れて精密にそろえる |
| **IQ-TREE** | 最尤法による系統樹推定 | 距離に要約せず、アミノ酸置換モデル(LG, WAG など)に基づく **確率的推論**。**ブートストラップ** で各枝の信頼度も算出 |

これらは Python ライブラリではなく **独立した実行ファイル(コマンドラインツール)** です。

### 10.1 MAFFT と IQ-TREE のインストール

環境に応じて以下のいずれかを実行してください(初回のみ)。

```bash
# Google Colab・Ubuntu/Debian
!apt-get install -y mafft iqtree

# conda 環境(より新しい IQ-TREE が手に入る)
conda install -c bioconda mafft iqtree

# macOS (Homebrew)
brew install mafft iqtree
```

> Note: IQ-TREE のコマンド名は環境により `iqtree`, `iqtree2`, `iqtree3` のいずれかです。
> 以下のコードでは自動検出し、バージョンに応じてオプションも切り替えます。

In [ ]:
# Colab で直接インストールしたいときは下のコメントを外す
# !apt-get install -y mafft iqtree

def find_command(candidates):
    for cmd in candidates:
        path = shutil.which(cmd)
        if path:
            return cmd, path
    return None, None

mafft_cmd, mafft_path = find_command(["mafft"])
iqtree_cmd, iqtree_path = find_command(["iqtree3", "iqtree2", "iqtree"])

print(f"MAFFT  : {mafft_path or '見つかりません'}")
print(f"IQ-TREE: {iqtree_path or '見つかりません'} (コマンド名: {iqtree_cmd})")

if not (mafft_cmd and iqtree_cmd):
    print("\n⚠ どちらかが無い場合は上の手順でインストールしてください。")
    print("  本格編はスキップされ、入門編(NJ)の結果のみ得られます。")

### 10.2 MAFFT で配列を多重整列

**MAFFT**(京都大学・加藤らが開発)は速くて精度の高い多重整列ツールです。
`--auto` で配列数とマシン性能に応じた最適アルゴリズムを自動選択します。

まず ID を整理した FASTA を書き出してから MAFFT に渡します。

In [ ]:
stripped_fasta = WORK_DIR / "gpcr_stripped.fasta"
clean_records = list(SeqIO.parse(FASTA_PATH, "fasta"))
for rec in clean_records:
    rec.id = rec.description.split("|")[0]
    rec.description = ""
SeqIO.write(clean_records, stripped_fasta, "fasta")
print(f"ID 整理済み FASTA: {stripped_fasta}")

aligned_path = WORK_DIR / "gpcr_aligned.fasta"

if mafft_cmd:
    print("\nMAFFT を実行中(数秒〜十数秒)...")
    with open(aligned_path, "w") as f:
        subprocess.run(
            [mafft_cmd, "--auto", "--quiet", str(stripped_fasta)],
            stdout=f, stderr=subprocess.PIPE, check=True, text=True,
        )
    print(f"完了: {aligned_path}")
else:
    print("MAFFT がインストールされていません。スキップします。")

### 10.3 多重整列の結果を確認

MAFFT は挿入・欠失(ギャップ `-`)を入れて全配列をそろえます。
配列長がバラバラ(348〜1078残基)だったので、整列後のカラム数はかなり長くなります
── これは短い配列に大量のギャップが入ったことを意味します。

In [ ]:
if aligned_path.exists():
    mafft_alignment = AlignIO.read(aligned_path, "fasta")
    print(f"配列数            : {len(mafft_alignment)}")
    print(f"カラム数(整列後)  : {mafft_alignment.get_alignment_length()}")
    print(f"元の配列長の範囲  : {seq_stats['length'].min()} 〜 {seq_stats['length'].max()} 残基")

    print("\n最初の60カラム × 6配列(ギャップ '-' に注目):")
    for rec in mafft_alignment[:6]:
        print(f"  {rec.id:8s} {str(rec.seq)[:60]}")
else:
    print("MAFFT 結果がないため、このセルはスキップします。")
    mafft_alignment = None

### 10.4 保存性プロファイル(どの位置がどれだけ保存されているか)

各カラムで「最も多いアミノ酸がどれだけ多数派か」= **保存性スコア** を計算し、配列に沿って描きます。

- 保存率の **高い** 領域 = 機能的に重要(7回膜貫通の骨格や、G タンパク質と結合する部位など)
- 保存率の **低い** 領域 = 種類ごとに多様(各受容体に固有のループや細胞外ドメイン)

GPCR は全長としては大きく違いますが、**膜貫通領域には保存された「アンカー」**が点在します。

In [ ]:
if mafft_alignment is not None:
    aln_array = np.array([list(str(rec.seq).upper()) for rec in mafft_alignment])
    n_seqs, n_cols = aln_array.shape

    conservation = np.zeros(n_cols)
    for j in range(n_cols):
        col = aln_array[:, j]
        col_no_gap = col[col != "-"]
        if len(col_no_gap) > 0:
            _, counts = np.unique(col_no_gap, return_counts=True)
            # ギャップだらけの列を過大評価しないよう、全配列数で正規化
            conservation[j] = counts.max() / n_seqs

    window = 30
    smoothed = np.convolve(conservation, np.ones(window) / window, mode="valid")

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(range(len(smoothed)), smoothed, color="steelblue", linewidth=1.2)
    ax.axhline(0.5, color="orange", linestyle="--", alpha=0.6, label="保存率 50%")
    ax.axhline(0.2, color="red", linestyle="--", alpha=0.6, label="保存率 20%")
    ax.set_xlabel(f"アラインメント上のカラム位置({window}カラム移動平均)")
    ax.set_ylabel("保存性スコア")
    ax.set_title(f"GPCR の保存性プロファイル({n_seqs}配列、{n_cols}カラム)")
    ax.set_ylim(0, 1.05)
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"\n保存率 ≥ 50% のカラム: {(conservation >= 0.5).sum()} / {n_cols} "
          f"({100*(conservation >= 0.5).mean():.1f}%)")
else:
    print("MAFFT 結果がないため、このセルはスキップします。")

### 10.5 最尤法と IQ-TREE

**最尤法 (Maximum Likelihood, ML)** は、配列を生み出す確率モデルに基づいて
「観察されたアラインメントを最もよく説明する系統樹」を統計的に探す方法です。

**距離法 (NJ) との違い**

| | 距離法 (NJ) | 最尤法 (IQ-TREE) |
| --- | --- | --- |
| 入力 | 距離行列(情報を要約) | アラインメント(全情報) |
| 置換モデル | なし、または固定 | **データから最適なモデルを自動選択**(LG, WAG, JTT + Γ など) |
| 計算量 | 軽い(数秒) | 重い(25配列で数十秒〜数分) |
| 信頼度 | 通常なし | **ブートストラップ**で各枝の支持率を算出 |

**IQ-TREE のオプション**

- `-s` : 入力アラインメント
- `-m MFP` : ModelFinder Plus(最適な **アミノ酸置換モデル** を自動選択)
- `-B 1000` (v2) / `-bb 1000` (v1) : UltraFast Bootstrap を 1000 回
- `-T AUTO` (v2) / `-nt AUTO` (v1) : スレッド数の自動設定
- `-redo` : 既存出力を上書き(再実行に便利)

ブートストラップとは、**アラインメントのカラムをランダムに重複抽出して系統樹を作り直す**ことを 1000 回繰り返し、
各枝が何 % の試行で再現されたかを「**支持率**」として返す仕組みです。

In [ ]:
ml_treefile = Path(str(aligned_path) + ".treefile")

if iqtree_cmd and aligned_path.exists():
    help_out = subprocess.run([iqtree_cmd, "--help"], capture_output=True, text=True)
    help_text = help_out.stdout + help_out.stderr
    is_modern = ("-B NUM" in help_text) or ("--prefix" in help_text)

    bootstrap_args = ["-B", "1000"] if is_modern else ["-bb", "1000"]
    threads_args = ["-T", "AUTO"] if is_modern else ["-nt", "AUTO"]
    print(f"IQ-TREE バージョン: {'v2/v3 系' if is_modern else 'v1 系'}")

    print("IQ-TREE を実行中(数十秒〜数分)...")
    proc = subprocess.run(
        [iqtree_cmd, "-s", str(aligned_path), "-m", "MFP",
         *bootstrap_args, *threads_args, "-redo"],
        capture_output=True, text=True,
    )
    if proc.returncode == 0:
        print("IQ-TREE 完了")
        print("\n生成されたファイル:")
        for f in sorted(WORK_DIR.glob("gpcr_aligned.fasta.*")):
            print(f"  {f.name}")
    else:
        print(f"IQ-TREE がエラー終了しました:\n{proc.stderr[-500:]}")
else:
    print("IQ-TREE またはアラインメントが無いため、スキップします。")

### 10.6 採用されたモデルを確認

ModelFinder は何十種類ものアミノ酸置換モデルから AIC/BIC で最良のものを選びます
(タンパク質では LG・WAG・JTT などが代表的)。`.iqtree` レポートで採用モデルを確認します。

In [ ]:
report_file = Path(str(aligned_path) + ".iqtree")
if report_file.exists():
    text = report_file.read_text()
    for keyword in ["Best-fit model", "Model of substitution",
                    "Log-likelihood of the tree",
                    "Akaike information criterion (AIC) score",
                    "Bayesian information criterion (BIC) score"]:
        for line in text.splitlines():
            if keyword in line:
                print(line.strip())
                break
else:
    print("IQ-TREE レポートがないため、このセルはスキップします。")

### 10.7 ML 系統樹の読み込みと描画

IQ-TREE の出力 `.treefile`(Newick 形式)を Biopython で読み込みます。
NJ と同じく明確な外群がないので **中点ルーティング** を使い、
内部ノードラベルに入っている **ブートストラップ支持率 (0〜100)** を取り出します。

In [ ]:
def load_ml_tree(path):
    tree = Phylo.read(path, "newick")
    tree.root_at_midpoint()
    for clade in tree.get_nonterminals():
        if clade.name:
            try:
                clade.confidence = float(clade.name)
            except ValueError:
                pass
        clade.name = None
    return tree

if ml_treefile.exists():
    ml_tree = load_ml_tree(ml_treefile)
    bs_values = [c.confidence for c in ml_tree.get_nonterminals()
                 if c.confidence is not None]
    print("ML 系統樹を読み込みました")
    print(f"  末端: {ml_tree.count_terminals()}")
    if bs_values:
        print(f"  ブートストラップ支持率: 平均 {np.mean(bs_values):.1f}, "
              f"範囲 [{min(bs_values):.0f}, {max(bs_values):.0f}]")
        print(f"  ≥ 95(強い支持)の枝: {sum(b >= 95 for b in bs_values)} / {len(bs_values)}")
else:
    print("ML 系統樹ファイルがないため、このセルはスキップします。")
    ml_tree = None

### 10.8 ML 系統樹をブートストラップ支持率付きで描画

各内部ノードに **ブートストラップ値** を表示します。

- 🟢 緑 (≥ 95): 強い支持(その枝はほぼ確実)
- 🟠 橙 (70–94): 中程度の支持
- 🔴 赤 (< 70): 弱い支持(その枝は信頼できない)

**支持率の低い枝は、データだけからは「どちらが正しいか決められなかった」**ことを意味します。

In [ ]:
def draw_ml_tree(tree, title, xlabel="アミノ酸置換数 / サイト"):
    # 末端の y 座標
    ypos = {tip: i for i, tip in enumerate(reversed(tree.get_terminals()))}
    def y_of(clade):
        if clade.is_terminal():
            return ypos[clade]
        return np.mean([y_of(c) for c in clade.clades])

    fig, ax = plt.subplots(figsize=(11, 11))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        Phylo.draw(tree, label_func=label_func, do_show=False, axes=ax,
                   label_colors=gene_color)

    # ブートストラップ値を内部ノードに重ねる
    depths = tree.depths()
    for clade in tree.get_nonterminals():
        if clade.confidence is None or clade == tree.root:
            continue
        bs = clade.confidence
        bc = "#1b9e77" if bs >= 95 else ("#d95f02" if bs >= 70 else "#b22222")
        ax.text(depths[clade], y_of(clade) + 1, f"{bs:.0f}",
                fontsize=8, ha="right", va="center", color=bc,
                bbox=dict(boxstyle="round,pad=0.1", facecolor="white",
                          edgecolor="none", alpha=0.8))

    ax.set_title(title, fontsize=13)
    ax.set_xlabel(xlabel)
    handles = [Patch(facecolor=c, label=class_name_ja[k])
               for k, c in class_to_color.items()]
    handles += [
        Patch(facecolor="#1b9e77", label="BS ≥ 95"),
        Patch(facecolor="#d95f02", label="BS 70–94"),
        Patch(facecolor="#b22222", label="BS < 70"),
    ]
    ax.legend(handles=handles, loc="lower right", framealpha=0.9, fontsize=9)
    plt.tight_layout()
    plt.show()

if ml_tree is not None:
    draw_ml_tree(ml_tree,
                 "GPCR の ML 系統樹(IQ-TREE + UltraFast Bootstrap × 1000)")
else:
    print("ML 系統樹がないため、このセルはスキップします。")

### 10.9 NJ と ML を並べて比較

同じデータから 2 つの方法で系統樹を作りました。並べて見比べてみましょう。

In [ ]:
if ml_tree is not None:
    fig, axes = plt.subplots(1, 2, figsize=(20, 11))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        Phylo.draw(nj_tree, label_func=label_func, do_show=False, axes=axes[0],
                   label_colors=gene_color)
        axes[0].set_title("NJ(ペアワイズ距離)")
        Phylo.draw(ml_tree, label_func=label_func, do_show=False, axes=axes[1],
                   label_colors=gene_color)
        axes[1].set_title("ML(MAFFT 整列 + IQ-TREE 最尤法 + UFBoot 1000)")
    plt.tight_layout()
    plt.show()
else:
    print("ML 系統樹がないため比較できません。")

**比較の観察ポイント**

- **大枠のクラス分け(A / B / C / F)** が両者で一致していれば、それは信頼できるシグナル
- **クラス A 内部の細かい枝**は方法によって少し違うことがある。ML 樹でその枝のブートストラップが低ければ、「データだけでは決められない部分」だったということ
- **枝の長さ**は ML のほうが情報量が多い(置換モデルでの推定値)── 同じ短い枝でも「ほとんど変わっていない」のか「最近分かれた」のかを区別できる

## 11. (発展)立体構造を眺めてみる

このデータセットには各受容体の **AlphaFold 予測立体構造**(`gpcr/pdb/`)も入っています。
配列はクラス間で大きく違っても、**「7回膜貫通」という骨格は共通**しているはずです。
`py3Dmol` がインストールされていれば、ノートブック内で 3D 構造をくるくる回して確認できます。

> 配列の系統樹で「遠い」とされた受容体どうしでも、立体構造の骨格は似ている ──
> これは **構造のほうが配列より進化的に保存されやすい** という重要な性質の表れです。

In [ ]:
# py3Dmol が無ければ: %pip install py3Dmol
try:
    import py3Dmol

    def show_structure(gene, color):
        pdb_files = list(PDB_DIR.glob(f"{gene}_*.pdb"))
        if not pdb_files:
            print(f"{gene} の PDB が見つかりません")
            return None
        view = py3Dmol.view(width=350, height=350)
        view.addModel(pdb_files[0].read_text(), "pdb")
        view.setStyle({"cartoon": {"color": color}})
        view.zoomTo()
        return view

    # クラス A(ロドプシン)とクラス C(代謝型グルタミン酸受容体)を見比べる
    print("RHO (クラスA, 348残基) ─ コンパクトな7回膜貫通")
    v1 = show_structure("RHO", "spectrum")
    if v1: v1.show()
    print("GRM1 (クラスC, 906残基) ─ 巨大な細胞外ドメイン + 7回膜貫通")
    v2 = show_structure("GRM1", "spectrum")
    if v2: v2.show()
except ImportError:
    print("py3Dmol が未インストールです。`%pip install py3Dmol` で導入すると 3D 表示できます。")
    print(f"PDB ファイルは {PDB_DIR} に {len(list(PDB_DIR.glob('*.pdb')))} 個あります。")

## 12. 発展編 — 立体構造に基づく系統樹推定

ここまでは **配列(文字列)** を比べてきました。しかし GPCR のようにクラス間で大きく分岐したファミリーでは、
揃えられるのは保存された膜貫通領域が中心で、アラインメント自体に不確実性が残りました。

> **アイデア:** タンパク質の **立体構造は配列よりも進化的に保存されやすい**。
> ならば、配列を揃える代わりに **構造そのものを重ね合わせて**、似ているかどうかを測れば、
> 配列ではぼやけてしまう遠縁の関係も捉えられるかもしれません。

**手順**

1. 各受容体の AlphaFold 構造(`gpcr/pdb/`)を **総当たり**で重ね合わせる(構造アラインメント)
2. 重なり具合を **TM-score**(0〜1、1 に近いほど同じ折りたたみ。おおむね 0.5 以上で「同じフォールド」)で数値化
3. **構造距離** $d = 1 - \mathrm{TMscore}$ を距離行列にして **NJ 法** で系統樹を作る
4. **配列の系統樹**と並べて、どこが一致し、どこが違うかを観察する

ここでは TM-align の Python 実装 **`tmtools`** を使います。
研究現場では **Foldseek / FoldTree / US-align / DALI** などがこの目的で広く使われています。

### 12.1 tmtools の準備

構造アラインメント用ライブラリ `tmtools`(TM-align の Python 実装)をインストールします。
構造の読み込みには既に入っている Biopython を使います。

In [ ]:
# 初回のみ(未インストールなら下のコメントを外す)
# %pip install tmtools

try:
    from tmtools import tm_align
    from tmtools.io import get_structure, get_residue_data
    tmtools_available = True
    print("tmtools: OK")
except ImportError:
    tmtools_available = False
    print("tmtools が未インストールです。`%pip install tmtools` を実行してください。")
    print("このセクションはスキップされます(系統樹は配列ベースの結果のみ)。")

### 12.2 構造の読み込みと総当たり構造比較

各 PDB ファイルから **Cα 原子の座標**と配列を取り出し、すべてのペアを TM-align で重ね合わせます。

TM-score は **短いほうの鎖で正規化** されるため、長さが違うと A→B と B→A で値が変わります(非対称)。
そこで **両方向の平均**を「構造類似度」とし、対称な距離行列を作ります。

> 25 構造 → 300 ペアの重ね合わせで、**おおよそ 2〜5 分**かかります(クラス C の巨大構造を含むため)。

In [ ]:
if tmtools_available and PDB_DIR.exists():
    # PDB から Cα 座標と配列を取得(ファイル名 ADRB2_AF-P07550.pdb -> 遺伝子名 ADRB2)
    struct_data = {}
    for pdb_path in sorted(PDB_DIR.glob("*.pdb")):
        gene = pdb_path.stem.split("_")[0]
        structure = get_structure(str(pdb_path))
        chain = next(structure.get_chains())
        coords, seq = get_residue_data(chain)
        struct_data[gene] = (coords, seq)
    struct_genes = sorted(struct_data)
    n_st = len(struct_genes)
    print(f"読み込んだ構造: {n_st}")

    # 総当たり TM-align(上三角だけ計算し、両方向の TM-score を平均して対称化)
    print("構造を総当たりで重ね合わせ中(2〜5分)...")
    sim = np.eye(n_st)
    n_pairs = n_st * (n_st - 1) // 2
    done = 0
    for i in range(n_st):
        ci, si = struct_data[struct_genes[i]]
        for j in range(i + 1, n_st):
            cj, sj = struct_data[struct_genes[j]]
            res = tm_align(ci, cj, si, sj)
            s = (res.tm_norm_chain1 + res.tm_norm_chain2) / 2
            sim[i, j] = sim[j, i] = s
            done += 1
        print(f"  {done}/{n_pairs} ペア完了", end="\r")
    print(f"\n完了。構造類似度行列: {n_st} × {n_st}")

    struct_dist = 1.0 - sim
    np.fill_diagonal(struct_dist, 0.0)

    g2i = {g: k for k, g in enumerate(struct_genes)}
    def sd(a, b):
        return struct_dist[g2i[a], g2i[b]]
    print("\n例(構造距離 = 1 - TMスコア):")
    print(f"  RHO vs OPN1SW (オプシン同士)   = {sd('RHO', 'OPN1SW'):.3f}")
    print(f"  RHO vs CASR  (クラスA vs クラスC) = {sd('RHO', 'CASR'):.3f}  ← 配列距離より近いことが多い")
    print(f"  SMO vs FZD1  (クラスF同士)      = {sd('SMO', 'FZD1'):.3f}")
else:
    print("tmtools または PDB フォルダが無いため、このセクションはスキップします。")
    struct_dist = None

### 12.3 構造類似度のヒートマップ

配列距離のヒートマップ(セクション6)と見比べてみましょう。
構造のほうが、クラスをまたいでも全体に「近い」(=同じ7回膜貫通フォールドを共有する)ことが見えるはずです。

In [ ]:
if struct_dist is not None:
    sdf = pd.DataFrame(struct_dist, index=struct_genes, columns=struct_genes)
    ordered_st = sorted(struct_genes,
                        key=lambda g: (class_order.index(gene_to_class[g]), g))
    sdf = sdf.loc[ordered_st, ordered_st]
    labels_st = [f"{g} ({gene_to_class[g]})" for g in sdf.index]

    fig, ax = plt.subplots(figsize=(11, 9))
    sns.heatmap(sdf, cmap="magma_r",
                xticklabels=labels_st, yticklabels=labels_st,
                cbar_kws={"label": "構造距離 (1 - TMスコア)"}, ax=ax, square=True)
    ax.set_title("25 受容体間の立体構造距離(クラス順に並べ替え)")
    plt.tight_layout()
    plt.show()
else:
    print("構造距離が無いため、このセルはスキップします。")

### 12.4 構造に基づく系統樹(NJ法)

構造距離行列から NJ 法で系統樹を作り、配列の樹と同じくクラスで色分けして描画します。

In [ ]:
if struct_dist is not None:
    lower_st = [[struct_dist[i, j] for j in range(i + 1)] for i in range(n_st)]
    struct_dm = DistanceMatrix(struct_genes, lower_st)
    struct_tree = DistanceTreeConstructor().nj(struct_dm)
    struct_tree.root_at_midpoint()
    for clade in struct_tree.get_nonterminals():
        clade.name = None

    draw_class_tree(struct_tree,
                    "GPCR の構造系統樹(TM-align + NJ、中点ルーティング)",
                    xlabel="構造距離 (1 - TMスコア)")
else:
    print("構造距離が無いため、このセルはスキップします。")
    struct_tree = None

### 12.5 配列の樹 vs 構造の樹

同じ 25 受容体について、**配列**から作った樹と **立体構造**から作った樹を並べます。
(配列側は、最尤樹があればそれを、無ければ入門編の NJ 樹を使います。)

In [ ]:
if struct_tree is not None:
    seq_tree = ml_tree if ("ml_tree" in dir() and ml_tree is not None) else nj_tree
    seq_label = ("配列: ML 樹 (IQ-TREE)" if seq_tree is ml_tree
                 else "配列: NJ 樹(ペアワイズ距離)")

    fig, axes = plt.subplots(1, 2, figsize=(20, 11))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        Phylo.draw(seq_tree, label_func=label_func, do_show=False, axes=axes[0],
                   label_colors=gene_color)
        axes[0].set_title(seq_label)
        Phylo.draw(struct_tree, label_func=label_func, do_show=False, axes=axes[1],
                   label_colors=gene_color)
        axes[1].set_title("構造: NJ 樹 (TM-align)")
    plt.tight_layout()
    plt.show()
else:
    print("構造系統樹が無いため比較できません。")

**比較の観察ポイント**

- **クラスの大枠(A/B/C/F)** は配列でも構造でも分かれているはず → どちらの情報にも強く刻まれた進化の信号
- **クラスをまたぐ距離**は構造のほうが小さい(=全体に似て見える)。これは「7回膜貫通」という共通フォールドが保存されているため
- **配列ではあいまいだった枝**が構造ではすっきり分かれる(またはその逆)ことがある。どの受容体で食い違うか探してみよう
- 配列と構造が食い違う部分は、**進化の途中で配列は大きく変わったが折りたたみは保たれた**(またはその逆)ことを示唆する

## 13. 考察 — 系統樹は何を語っているか

上で得た 2 つの樹(NJ と ML)を見比べながら、以下の問いを考えてみましょう。

### Q1. GPCR の 4 クラス(A/B/C/F)は再現できた?

配列だけから作った樹で、クラス B(GCGR, GLP1R, PTH1R)・C(GRM1, CASR)・F(SMO, FZD1)は
それぞれ 1 つの枝にまとまりましたか? ML 樹で、それを支える枝のブートストラップ支持率は?

### Q2. クラス A の中の「機能グループ」は見えた?

クラス A はヒト GPCR の大半を占める巨大グループです。その内部を見ると…

- **オプシン**(RHO, OPN1SW, OPN1MW)── 光を受容する。3 つは近い?
- **アミン受容体**(ADRB1/2, ADRA1A/2A, DRD1/2, HTR1A/2A, CHRM1/2, HRH1)──
  アドレナリン・ドーパミン・セロトニン・ヒスタミン・アセチルコリンなどの **小さなアミン分子** を受容。まとまっている?
- **ペプチド/脂質受容体**(OPRM1, CNR1, AGTR1, CXCR4)── 大きめのリガンドを受容

「同じ種類のリガンドを受容する受容体は系統的にも近い」傾向が見えるはずです。

### Q3. 機能が似ていても系統的に遠い例はある?

- クラス B の **GLP1R / GCGR / PTH1R** はすべてペプチドホルモン受容体ですが、クラス A のペプチド受容体(OPRM1 など)とは
  系統的に **遠い** はずです。「ペプチドを受容する」という機能は、**別々の系統で独立に獲得**された(収斂)可能性があります。
- クラス F の **SMO / FZD1** は発生シグナルに関わり、他クラスとは大きく離れています。

### Q4. 枝の長さは何を語る?

- 短い枝 → その受容体の配列は祖先と似たまま(保存的)
- 長い枝 → 多くのアミノ酸置換が起きた(速く進化した、または孤立して長く別系統)

クラス C の受容体(GRM1, CASR)が長い枝の先にあるなら、それは「他クラスから早く分かれ、独自に大きく変化した」ことを示唆します。

### Q5. ブートストラップ支持率が低い枝はなぜ低い?

- 配列に含まれる「系統の信号」が弱い(短期間に複数系統が連続して分かれた等)
- アラインメントできる保存領域が短い(GPCR は膜貫通部しか揃わない)
- 進化速度が極端に違う系統が混じる **長枝誘引 (long branch attraction)**

支持率の低い部分は「**データだけでは決められなかった**」と素直に受け止め、別の情報(立体構造・機能)で補強します。

### Q6. 配列の樹と構造の樹はどこが違った?

セクション12で作った **構造系統樹** と、配列の樹(NJ / ML)を見比べてみましょう。

- 大枠のクラス分け(A/B/C/F)は **両方で一致**していましたか?
- **一致しなかった枝**はどこ? その受容体は「配列は大きく変わったが折りたたみは保たれた」のか、それとも「予測構造の不確実な領域に引っ張られた」のか、どちらだと思いますか?
- 配列・構造・機能(リガンドの種類)の 3 つの情報が **そろって支持する**関係は、最も信頼できる進化的シグナルと言えます。今回そのような関係はありましたか?

## 14. この解析の限界と注意点

MAFFT + IQ-TREE は研究レベルのパイプラインですが、それでも完全ではありません。

1. **配列と構造は別の側面を見ている**
    - 配列では全長の半分以上が違うことも多く、揃えられるのは保存された膜貫通領域が中心
    - セクション12で見たように **立体構造**は配列より保存されやすく、別の絵を見せてくれる ── ただし AlphaFold は **予測**構造なので、予測の不確実な領域(長いループなど)は重ね合わせの誤差になりうる

2. **クラスをまたぐアラインメントの難しさ**
    - A/B/C/F はあまりに分岐していて、整列の「同じ位置」の対応づけ自体に不確実性がある
    - その不確実性がブートストラップ支持率の低さに表れる

3. **アミノ酸置換モデルの限界**
    - ModelFinder は最良モデルを選ぶが、現実の進化が完全にモデル通りとは限らない
    - 膜貫通領域(疎水性)と細胞外領域では置換の傾向が異なる(本来は領域ごとに別モデルが理想)

4. **サンプリングの偏り**
    - 今回はクラス A が 18、B が 3、C が 2、F が 2 と偏っている
    - ヒトの GPCR だけを選んでおり、他の生物や Adhesion(接着)クラスは含まれていない

## 15. 発展課題

1. **クラスごとに解析してみる**
    - クラス A の 18 受容体だけを取り出して系統樹を作ると、機能グループ(オプシン・アミン・ペプチド)の関係がよりくっきり見えるはず。

2. **距離モデルを変える**
    - 入門編の `BLOSUM62` を `BLOSUM45`(遠縁向け)や `PAM250` に変えると距離行列・NJ 樹はどう変わる?

3. **整列を可視化する**
    - `gpcr_aligned.fasta` を [Jalview](https://www.jalview.org/) や [AliView](https://ormbunkar.se/aliview/) で開き、
      保存性プロファイルで見えた「保存カラム」が膜貫通領域に対応しているか確かめてみよう。

4. **別のツールで構造系統樹を作る**
    - セクション12では `tmtools` で構造系統樹を作った。同じことを [Foldseek](https://github.com/steineggerlab/foldseek) や
      [FoldTree](https://github.com/DessimozLab/fold_tree)、`US-align` でも試し、ツールによって樹が変わるか比べてみよう。
    - TM-score の代わりに **RMSD** や **LDDT** を距離に使うと、樹はどう変わる?

5. **本物のリファレンスと比べる**
    - [GPCRdb](https://gpcrdb.org/) には GPCR の系統樹や分類が整理されている。今回作った樹と一致する? しない枝はどこ?

6. **DNA の系統樹(鳥類・哺乳類ノートブック)と比較する**
    - 「DNA で種を比べる」のと「タンパク質で遺伝子ファミリーを比べる」のは、アラインメント・距離・モデルの考え方がどう違ったか整理してみよう。